# Oracle Best MSE Encoder vs Random

For a chosen MSE-embedding ablation, this notebook selects the **oracle best encoder** per `(task, subset)` by averaging each encoder policy over its 10 permutation runs, then picking the best encoder among the five encoder-backed policies. It then computes the **paired difference vs random** and bootstraps the dataset-level mean and 95% CI via the standard `subset -> task -> dataset` hierarchy.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FormatStrFormatter, MultipleLocator

REPO_ROOT = Path('/data/ddmg/mvseg-ordering')
for p in [REPO_ROOT, REPO_ROOT / 'UniverSeg', REPO_ROOT / 'MultiverSeg', REPO_ROOT / 'ScribblePrompt']:
    if str(p) not in sys.path:
        sys.path.append(str(p))

from experiments.analysis.hierarchical_ci import (
    dataset_bootstrap_stats,
    hierarchical_bootstrap_task_estimates,
)
from experiments.analysis.notebook_plot_style import (
    BAR_Y_TICK_STEP_DICE,
    BAR_Y_TICK_STEP_ITERS,
    FONT_SIZE_PANEL_TITLE,
    FONT_SIZE_SUPTITLE,
    FONT_SIZE_SUPYLABEL,
    FONT_SIZE_XTICKS,
    FONT_SIZE_YTICKS,
)
from experiments.analysis.task_explorer import FAMILY_ROOTS


In [ ]:
# -----------------------------
# Config
# -----------------------------
PROCEDURE = 'random_v_MSE_embedding_final'
METRICS = ['initial_dice', 'iterations_used', 'final_dice']

METRIC_DISPLAY = {
    'initial_dice': 'Initial Dice Score',
    'final_dice': 'Final Dice Score',
    'iterations_used': 'Interactions Used',
}

# Choose one MSE-embedding ablation. If None, defaults to context_mean when present.
ABLATION = 'pretrained_baseline5p_mse_embedding_context_mean'

BASELINE_POLICY = 'random'
ORACLE_POLICY_NAME = 'oracle_best_encoder'
FAMILY_DISPLAY = {'T1mix': 'COBRE'}

# Optional: restrict encoders manually. Set to None to use all non-random policies under the ablation.
ENCODER_POLICIES = None
ENCODER_DISPLAY = {
    'clip': 'CLIP',
    'dinov2': 'DINOv2',
    'medsam': 'MedSAM',
    'multiverseg': 'MultiverSeg',
    'vit': 'ViT',
}

N_BOOT = 2000
SEED = 23
N_COLS = 3

SAVE_FIG = False
FIG_DIR = REPO_ROOT / 'figures' / 'mse_embedding_oracle_best_encoder_vs_random'
ZOOM_PAD_FRAC = 0.20


In [ ]:
def load_planb_for_procedure(
    repo_root: Path,
    procedure: str,
    *,
    filename: str = 'subset_support_images_summary.csv',
) -> pd.DataFrame:
    scripts_root = repo_root / 'experiments' / 'scripts' / procedure
    frames: list[pd.DataFrame] = []

    for root_name, family in FAMILY_ROOTS.items():
        family_root = scripts_root / root_name
        if not family_root.exists():
            continue

        for task_dir in sorted(p for p in family_root.iterdir() if p.is_dir()):
            task_id = f'{family}/{task_dir.name}'

            for ablation_dir in sorted(p for p in task_dir.iterdir() if p.is_dir()):
                policy_dirs = sorted(
                    p for p in ablation_dir.iterdir()
                    if p.is_dir() and (p / 'B' / filename).exists()
                )

                if policy_dirs:
                    for policy_dir in policy_dirs:
                        csv_path = policy_dir / 'B' / filename
                        df = pd.read_csv(csv_path)
                        if df.empty:
                            continue
                        df['family'] = family
                        df['task_id'] = task_id
                        df['task_name'] = task_dir.name
                        df['ablation'] = ablation_dir.name
                        df['policy_name'] = policy_dir.name
                        df['__source__'] = str(csv_path)
                        frames.append(df)
                    continue

                csv_path = ablation_dir / 'B' / filename
                if csv_path.exists():
                    df = pd.read_csv(csv_path)
                    if df.empty:
                        continue
                    df['family'] = family
                    df['task_id'] = task_id
                    df['task_name'] = task_dir.name
                    df['ablation'] = ablation_dir.name
                    df['policy_name'] = 'random'
                    df['__source__'] = str(csv_path)
                    frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f'No Plan B summaries found under experiments/scripts/{procedure}/**/{filename}'
        )

    return pd.concat(frames, ignore_index=True)


def is_higher_better(metric_name: str) -> bool:
    return metric_name not in {'iterations_used'}


def encoder_display_name(policy_name: str) -> str:
    name = str(policy_name)
    if name == BASELINE_POLICY:
        return 'Random'
    suffix = name.rsplit('_', 1)[-1]
    return ENCODER_DISPLAY.get(suffix, suffix)


def build_subset_scores_oracle_best_encoder(
    family_df: pd.DataFrame,
    *,
    metric: str,
    encoder_policies: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    required = {'task_id', 'subset_index', 'policy_name', 'permutation_index', metric}
    missing = required - set(family_df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    per_perm = (
        family_df
        .groupby(['task_id', 'subset_index', 'policy_name', 'permutation_index'], as_index=False)[metric]
        .mean()
        .rename(columns={metric: 'perm_score'})
    )

    per_subset = (
        per_perm
        .groupby(['task_id', 'subset_index', 'policy_name'], as_index=False)['perm_score']
        .mean()
        .rename(columns={'perm_score': 'subset_score'})
    )

    baseline_subset = (
        per_subset[per_subset['policy_name'].astype(str) == BASELINE_POLICY]
        [['task_id', 'subset_index', 'subset_score']]
        .rename(columns={'subset_score': 'baseline_subset_score'})
    )

    encoder_subset = per_subset[per_subset['policy_name'].astype(str).isin(encoder_policies)].copy()
    if encoder_subset.empty:
        empty = pd.DataFrame(columns=['task_id', 'subset_index', 'oracle_policy_name', 'oracle_subset_score'])
        return baseline_subset, empty

    use_max = is_higher_better(metric)
    encoder_subset = encoder_subset.sort_values(
        ['task_id', 'subset_index', 'subset_score', 'policy_name'],
        ascending=[True, True, not use_max, True],
        kind='mergesort',
    )
    oracle_subset = (
        encoder_subset
        .groupby(['task_id', 'subset_index'], as_index=False)
        .first()
        .rename(columns={
            'policy_name': 'oracle_policy_name',
            'subset_score': 'oracle_subset_score',
        })
    )
    return baseline_subset, oracle_subset


In [ ]:
raw_df = load_planb_for_procedure(REPO_ROOT, PROCEDURE)
print(f'Loaded rows: {len(raw_df):,}')
print('Families:', sorted(raw_df['family'].unique().tolist()))
print('Ablations:', sorted(raw_df['ablation'].astype(str).unique().tolist()))

available_ablations = sorted(raw_df['ablation'].astype(str).unique().tolist())
if ABLATION is None:
    preferred = 'pretrained_baseline5p_mse_embedding_context_mean'
    active_ablation = preferred if preferred in available_ablations else available_ablations[0]
    print(f'[info] Using ablation: {active_ablation}')
else:
    active_ablation = str(ABLATION)
    if active_ablation not in available_ablations:
        raise ValueError(f'ABLATION={active_ablation!r} not in available ablations: {available_ablations}')

df = raw_df[raw_df['ablation'].astype(str) == active_ablation].copy()
metrics_to_run = [m for m in METRICS if m in df.columns]
if not metrics_to_run:
    raise ValueError(f'None of METRICS are available for {active_ablation}. Requested={METRICS}')

available_policies = sorted(df['policy_name'].astype(str).unique().tolist())
if ENCODER_POLICIES is None:
    encoder_policies = [p for p in available_policies if p != BASELINE_POLICY]
else:
    encoder_policies = [p for p in ENCODER_POLICIES if p in available_policies]

if not encoder_policies:
    raise ValueError('No encoder policies available after filtering.')

print('Using ablation:', active_ablation)
print('Policies kept for oracle encoder selection:', encoder_policies)
print('Metrics to run:', metrics_to_run)


In [ ]:
def build_summary_for_metric(metric_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    pick_rows: list[pd.DataFrame] = []

    for family, fam_df in df.groupby('family'):
        baseline_subset, oracle_subset = build_subset_scores_oracle_best_encoder(
            fam_df,
            metric=metric_name,
            encoder_policies=encoder_policies,
        )
        if baseline_subset.empty or oracle_subset.empty:
            print(f'[warn] family={family}: missing baseline or encoder subset scores for metric={metric_name}; skipping')
            continue

        paired = oracle_subset.merge(
            baseline_subset,
            on=['task_id', 'subset_index'],
            how='inner',
            validate='one_to_one',
        )
        if paired.empty:
            continue

        paired['delta'] = paired['oracle_subset_score'] - paired['baseline_subset_score']
        subset_scores_by_task = {
            str(task_id): grp['delta'].to_numpy(dtype=float)
            for task_id, grp in paired.groupby('task_id')
        }

        task_boot = hierarchical_bootstrap_task_estimates(
            subset_scores_by_task,
            n_boot=N_BOOT,
            seed=SEED,
        )
        _, mean, ci_lo, ci_hi = dataset_bootstrap_stats(task_boot, alpha=0.05)

        rows.append(
            {
                'family': str(family),
                'ablation': active_ablation,
                'metric': metric_name,
                'policy_name': ORACLE_POLICY_NAME,
                'mean': float(mean),
                'ci_lo': float(ci_lo),
                'ci_hi': float(ci_hi),
                'n_tasks': int(paired['task_id'].nunique()),
                'n_subsets': int(len(paired)),
            }
        )

        picks = paired[['task_id', 'subset_index', 'oracle_policy_name']].copy()
        picks['family'] = family
        picks['metric'] = metric_name
        pick_rows.append(picks)

    summary = pd.DataFrame(rows)
    picks = pd.concat(pick_rows, ignore_index=True) if pick_rows else pd.DataFrame()
    return summary, picks


summary_by_metric: dict[str, pd.DataFrame] = {}
picks_by_metric: dict[str, pd.DataFrame] = {}
for metric_name in metrics_to_run:
    summary, picks = build_summary_for_metric(metric_name)
    summary_by_metric[metric_name] = summary
    picks_by_metric[metric_name] = picks
    print(f'[{metric_name}] rows={len(summary)}')
    if not summary.empty:
        display(summary)
    if not picks.empty:
        pick_counts = (
            picks.assign(encoder_label=picks['oracle_policy_name'].map(encoder_display_name))
            .groupby(['family', 'encoder_label'], as_index=False)
            .size()
            .rename(columns={'size': 'n_subset_picks'})
        )
        display(pick_counts)


In [ ]:
def plot_dataset_bar_grid(summary_df: pd.DataFrame) -> None:
    if summary_df.empty:
        print('No summary rows to plot.')
        return

    metric_name = str(summary_df['metric'].iloc[0])
    metric_label = METRIC_DISPLAY.get(metric_name, metric_name.replace('_', ' ').title())
    families = sorted(summary_df['family'].unique().tolist())

    n_families = len(families)
    n_cols = max(1, int(N_COLS))
    n_rows = int(math.ceil(n_families / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.0 * n_cols, 3.8 * n_rows),
        squeeze=False,
        sharey=True,
    )
    axes_flat = axes.ravel()

    global_lo = float(summary_df['ci_lo'].min())
    global_hi = float(summary_df['ci_hi'].max())
    global_span = max(global_hi - global_lo, 1e-8)
    global_pad = ZOOM_PAD_FRAC * global_span
    y_label = f'Δ {metric_label} vs. Random'
    fig.supylabel(y_label, fontsize=FONT_SIZE_SUPYLABEL, x=0.02)

    for i, family in enumerate(families):
        ax = axes_flat[i]
        fam = summary_df[summary_df['family'] == family].copy().reset_index(drop=True)
        if fam.empty:
            ax.axis('off')
            continue

        means = fam['mean'].to_numpy(dtype=float)
        ci_lo = fam['ci_lo'].to_numpy(dtype=float)
        ci_hi = fam['ci_hi'].to_numpy(dtype=float)
        err_lo = means - ci_lo
        err_hi = ci_hi - means
        x = np.arange(len(fam), dtype=float)

        ax.bar(x, means, color='#4C78A8', alpha=0.85, width=0.72)
        ax.errorbar(x, means, yerr=np.vstack([err_lo, err_hi]), fmt='none', ecolor='black', capsize=3, linewidth=1)
        ax.axhline(0.0, color='black', linestyle='--', linewidth=1.0, alpha=0.8)
        ax.set_ylim(global_lo - global_pad, global_hi + global_pad)
        ax.tick_params(axis='y', labelleft=True)

        tick_cfg = {
            'iterations_used': (BAR_Y_TICK_STEP_ITERS, '%.1f'),
            'initial_dice': (BAR_Y_TICK_STEP_DICE, '%.2f'),
            'final_dice': (BAR_Y_TICK_STEP_DICE, '%.2f'),
        }
        tick_step, tick_fmt = tick_cfg.get(metric_name, (BAR_Y_TICK_STEP_DICE, '%.2f'))
        if tick_step is not None:
            ax.yaxis.set_major_locator(MultipleLocator(tick_step))
            ax.yaxis.set_major_formatter(FormatStrFormatter(tick_fmt))

        ax.set_title(FAMILY_DISPLAY.get(family, family), fontsize=FONT_SIZE_PANEL_TITLE)
        ax.set_xticks(x)
        ax.tick_params(axis='y', labelsize=FONT_SIZE_YTICKS)
        ax.set_xticklabels(['oracle best encoder'], rotation=20, ha='right', fontsize=FONT_SIZE_XTICKS)
        ax.grid(axis='y', linestyle='--', alpha=0.35)

    for j in range(n_families, len(axes_flat)):
        axes_flat[j].axis('off')

    fig.suptitle(
        f'Paired delta of oracle best MSE encoder vs random on {metric_label.lower()}',
        fontsize=FONT_SIZE_SUPTITLE,
        y=0.995,
    )
    fig.tight_layout(rect=(0.03, 0, 1, 0.97))

    if SAVE_FIG:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        out_path = FIG_DIR / f'{PROCEDURE.replace("/", "_")}_{active_ablation}_{metric_name}_oracle_best_encoder_vs_random_bar_grid.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        print(f'saved: {out_path}')

    plt.show()


for metric_name in metrics_to_run:
    metric_summary = summary_by_metric.get(metric_name)
    if metric_summary is None or metric_summary.empty:
        continue
    plot_dataset_bar_grid(metric_summary)
